# Product Failure Prediction — a low-signal, distribution-shift problem

ITI Intake 46 course competition, on the Kaggle *Tabular Playground Series — August 2022*
dataset. A manufacturer measures each unit of a product; predict the probability a unit
fails quality control.

| | |
|---|---|
| Metric | ROC-AUC |
| Train / test | 26,570 / 20,775 rows, 24 features |
| Private leaderboard | **0.58974** (6th) |

**The headline number is not broken.** The maximum achievable AUC on this dataset is
around 0.59. The signal is genuinely that weak, and that single fact drives every
decision below — which model class wins, which "improvements" are real, and how much of
the leaderboard is noise.

This notebook is the cleaned-up narrative. The raw exploration log, with every dead end
in the order I hit them, is in `02_experiments_log.ipynb`.

In [1]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

# Set RUN_SEARCH=1 to re-run the Optuna searches and the full architecture sweep.
# They take a while and land on the parameters already hard-coded below.
RUN_SEARCH = bool(int(os.environ.get("RUN_SEARCH", "0")))

# Point this at the directory holding train.csv / test.csv / sample_submission.csv.
DATA_DIR = Path(os.environ.get("DATA_DIR", ".."))

SEED = 42
np.random.seed(SEED)
print(f"data: {DATA_DIR.resolve()}   RUN_SEARCH={RUN_SEARCH}")

data: /home/sherif/Python Codes/Machine Learning Course Competition/Competition 2   RUN_SEARCH=False


## 1. The data, and the one property that matters

`product_code` is the whole problem. Train contains codes A–E; test contains F–I. **Every
product code in the test set is absent from training.** Anything the model learns that is
specific to a product cannot transfer.

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print(f"train {train.shape}   test {test.shape}")
print(f"train product_code: {sorted(train.product_code.unique())}")
print(f"test  product_code: {sorted(test.product_code.unique())}")
print(f"overlap: {sorted(set(train.product_code) & set(test.product_code)) or 'none'}")
print(f"\nfailure rate: {train.failure.mean():.4f}")

train (26570, 26)   test (20775, 25)
train product_code: ['A', 'B', 'C', 'D', 'E']
test  product_code: ['F', 'G', 'H', 'I']
overlap: none

failure rate: 0.2126


In [3]:
# Missingness is scattered through most measurement columns, in both splits.
miss = pd.DataFrame({
    "train_%": train.isna().mean().mul(100).round(2),
    "test_%": test.isna().mean().mul(100).round(2),
})
print(miss[miss.sum(axis=1) > 0].to_string())

                train_%  test_%
loading            0.94    1.07
measurement_10     4.89    5.14
measurement_11     5.53    5.47
measurement_12     6.03    5.97
measurement_13     6.68    6.27
measurement_14     7.05    6.93
measurement_15     7.56    7.42
measurement_16     7.94    8.08
measurement_17     8.60    8.38
measurement_3      1.43    1.58
measurement_4      2.02    1.97
measurement_5      2.54    2.45
measurement_6      3.00    3.00
measurement_7      3.53    3.47
measurement_8      3.94    4.07
measurement_9      4.62    4.35


## 2. Validation, before anything else

Because the test product codes are unseen, a random split flatters you: rows from product
A appear on both sides of the split, so the model can memorise A-specific quirks and get
credit for it. The honest proxy is **`StratifiedGroupKFold` grouped on `product_code`** —
each fold holds out an entire product, so validation always measures generalisation to an
*unseen product*.

Below, the same model is scored both ways. This comparison is the most important cell in
the notebook.

In [4]:
FEATURES_RAW = [c for c in train.columns if c.startswith("measurement")] + ["loading"]

def quick_pipeline():
    return make_pipeline(SimpleImputer(strategy="mean"), StandardScaler(),
                         LogisticRegression(C=0.01, solver="liblinear", random_state=SEED))

X_raw = train[FEATURES_RAW]
y = train["failure"].values
groups = train["product_code"].values

grouped = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
random_ = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

g_scores = cross_val_score(quick_pipeline(), X_raw, y, groups=groups, cv=grouped, scoring="roc_auc")
r_scores = cross_val_score(quick_pipeline(), X_raw, y, cv=random_, scoring="roc_auc")

print(f"grouped on product_code : {g_scores.mean():.5f}  (std {g_scores.std():.5f})")
print(f"plain random split      : {r_scores.mean():.5f}  (std {r_scores.std():.5f})")
print(f"optimism from the random split: {r_scores.mean() - g_scores.mean():+.5f}")

grouped on product_code : 0.58743  (std 0.00436)
plain random split      : 0.58747  (std 0.00707)
optimism from the random split: +0.00005


### The answer is: it doesn't matter here — and that is worth knowing

The two splits agree to about 5e-5, against a fold standard deviation of ~0.007. Holding
out an entire product costs the model essentially nothing.

The reason is visible in the data. The per-unit `measurement_*` columns carry almost no
product-specific structure, and the only genuinely product-level fields
(`attribute_0`–`attribute_3`) are constant within a code and get dropped in feature
selection. There is nothing product-specific left to memorise.

This is a negative result, and I am keeping it rather than deleting it. My original
notebook built the grouped CV, then switched to a plain `StratifiedKFold` from the
model-comparison cell onward and never switched back — and every table in the README was
reported as though it were grouped. Re-running under the grouped split is how I know the
conclusions survive. Had they not, I would have had a README full of numbers from an
analysis I never ran.

The grouped split is still the right design: it is the only one that *could* expose
leakage, and you cannot know in advance there is none to expose. Everything from here
down uses it.

## 3. Preprocessing — transductive, and identical on both sides

Two decisions:

- **`loading` is log-transformed on train and test together.** Concatenate, transform
  once, split back. See §7 for what happens when you don't.
- **Interactions with `measurement_13`.** SHAP interaction values on an XGBoost probe
  identified `measurement_13` as a hub feature, so it gets explicit products with the
  other strong measurements.

The final feature set is 10 main effects plus 3 interactions — 13 columns out of 24.
L1-penalised logistic regression zeroed out `measurement_1`, `_3`, `_6`, `_8`, `_11`,
`_12`, `_15`, `_16` outright, so they are dropped.

In [5]:
MAIN = ["loading", "measurement_17", "measurement_13", "measurement_5", "measurement_14",
        "measurement_9", "measurement_10", "measurement_4", "measurement_7", "measurement_2"]
INTER = ["inter_13_10", "inter_13_load", "inter_13_17"]
FEATURES = MAIN + INTER

def build_features(train_df, test_df):
    """Concatenate, transform once, split back -- so both sides get identical treatment."""
    tr, te = train_df.copy(), test_df.copy()
    tr["is_train"], te["is_train"] = 1, 0
    cols = [c for c in tr.columns if c in te.columns]
    full = pd.concat([tr[cols], te[cols]], axis=0, ignore_index=True)

    full["loading"] = np.log(full["loading"])

    full["inter_13_10"] = full["measurement_13"] * full["measurement_10"]
    full["inter_13_load"] = full["measurement_13"] * full["loading"]
    full["inter_13_17"] = full["measurement_13"] * full["measurement_17"]

    out_tr = full[full.is_train == 1][FEATURES].reset_index(drop=True)
    out_te = full[full.is_train == 0][FEATURES].reset_index(drop=True)
    return out_tr, out_te

X, X_test = build_features(train, test)
print(f"X {X.shape}   X_test {X_test.shape}")
print(f"loading mean -- train {X.loading.mean():.4f}  test {X_test.loading.mean():.4f}")

X (26570, 13)   X_test (20775, 13)
loading mean -- train 4.8062  test 4.8043


## 4. Linear models beat trees

With the grouped split in place, gradient boosting loses — consistently. When the true
signal is weak and close to linear, trees spend their capacity carving up noise; a
strongly regularised linear model cannot. The Optuna search over logistic regression
converged to `C ≈ 0.0074`, which is "regularise almost everything away" — exactly right
here.

In [6]:
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

def grouped_cv(model, X, y, groups, n_splits=5):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    scores = []
    for tr_i, va_i in cv.split(X, y, groups):
        m = model()
        m.fit(X.iloc[tr_i], y[tr_i])
        scores.append(roc_auc_score(y[va_i], m.predict_proba(X.iloc[va_i])[:, 1]))
    return np.array(scores)

# C from the Optuna search; re-run it with RUN_SEARCH=1.
BEST_C = 0.007428212018814894

candidates = {
    "Logistic regression": lambda: make_pipeline(
        SimpleImputer(strategy="mean"), StandardScaler(),
        LogisticRegression(C=BEST_C, solver="lbfgs", max_iter=2000, random_state=SEED)),
    "XGBoost": lambda: make_pipeline(
        SimpleImputer(strategy="mean"),
        XGBClassifier(n_estimators=272, max_depth=3, learning_rate=0.0063,
                      subsample=0.595, colsample_bytree=0.851, reg_alpha=3.13,
                      reg_lambda=6.87, eval_metric="auc", random_state=SEED)),
    "CatBoost": lambda: make_pipeline(
        SimpleImputer(strategy="mean"),
        CatBoostClassifier(iterations=400, depth=4, learning_rate=0.02,
                           verbose=0, random_seed=SEED)),
}

rows = []
for name, factory in candidates.items():
    s = grouped_cv(factory, X, y, groups)
    rows.append({"model": name, "mean AUC": round(s.mean(), 5), "std": round(s.std(), 5)})
    print(f"{name:22} {s.mean():.5f}  (std {s.std():.5f})")

pd.DataFrame(rows).sort_values("mean AUC", ascending=False)

Logistic regression    0.58939  (std 0.00475)


XGBoost                0.58746  (std 0.00586)


CatBoost               0.58659  (std 0.00547)


,model,mean AUC,std
0,Logistic regression,0.58939,0.00475
1,XGBoost,0.58746,0.00586
2,CatBoost,0.58659,0.00547


## 5. Imputation didn't matter

Three strategies, all differences well inside one standard deviation. I kept
`SimpleImputer`. This is the first of several "improvements" that the noise floor killed.

In [7]:
imputers = {"Simple mean": SimpleImputer(strategy="mean"),
            "KNN (k=5)": KNNImputer(n_neighbors=5)}

for name, imp in imputers.items():
    factory = lambda imp=imp: make_pipeline(
        imp, StandardScaler(),
        LogisticRegression(C=BEST_C, solver="lbfgs", max_iter=2000, random_state=SEED))
    s = grouped_cv(factory, X, y, groups)
    print(f"{name:14} {s.mean():.5f}  (std {s.std():.5f})")

print("\nIterative (MICE) in the original run: 0.5823 (std 0.0076) -- also inside the noise.")

Simple mean    0.58939  (std 0.00475)


KNN (k=5)      0.58948  (std 0.00476)

Iterative (MICE) in the original run: 0.5823 (std 0.0076) -- also inside the noise.


## 6. Read the noise floor

The fold standard deviation is around 0.007. **Differences smaller than roughly 0.014 AUC
are not real.** Applying that rule kills the entire imputation comparison above, several
architecture "wins" below, and — as it turns out — most of the leaderboard.

In [8]:
FOLD_STD = 0.007
board = pd.DataFrame({
    "rank": [1, 2, 3, 4, 5, 6],
    "AUC": [0.59179, 0.59113, 0.59092, 0.59074, 0.59016, 0.58974],
})
board["gap_to_1st"] = (board.AUC.iloc[0] - board.AUC).round(5)
board["inside_noise"] = board.gap_to_1st < FOLD_STD
print(board.to_string(index=False))
print(f"\nfull spread 1st..6th: {board.AUC.iloc[0] - board.AUC.iloc[-1]:.5f}")
print(f"one fold std        : {FOLD_STD:.5f}")

 rank     AUC  gap_to_1st  inside_noise
    1 0.59179     0.00000          True
    2 0.59113     0.00066          True
    3 0.59092     0.00087          True
    4 0.59074     0.00105          True
    5 0.59016     0.00163          True
    6 0.58974     0.00205          True

full spread 1st..6th: 0.00205
one fold std        : 0.00700


## 7. The bug worth recording

I log-transformed `loading` in train but not in test. The `measurement_13 x loading`
interaction was then computed on values about 27x larger on one side.

**Nothing caught it.** Not the CV score, not a crash, not a warning — the model trained
fine and the AUC looked plausible. The only visible symptom was that predicted
probabilities came out around 0.02 instead of the expected ~0.21.

The fix is structural (transform on the concatenation, as in §3) plus a cheap assertion
before any submission is written.

In [9]:
def demo_the_bug():
    tr, te = train.copy(), test.copy()
    tr["loading"] = np.log(tr["loading"])      # train only -- the bug
    ratio = te["loading"].mean() / tr["loading"].mean()
    print(f"loading mean -- train {tr.loading.mean():.3f}  test {te.loading.mean():.3f}"
          f"   ratio {ratio:.1f}x")

demo_the_bug()

def check_submission(preds, expected=0.21, tol=0.05):
    """The cheapest bug detector I have. Run before writing any submission."""
    m = float(np.mean(preds))
    assert abs(m - expected) < tol, f"mean prediction {m:.4f} is far from {expected}"
    print(f"prediction mean {m:.4f} -- ok")

loading mean -- train 4.806  test 127.635   ratio 26.6x


## 8. RankGauss + swish

`QuantileTransformer(output_distribution="normal")` — RankGauss — maps each feature to a
Gaussian by rank. It suits a near-linear decision boundary and is robust to the skew in
`loading`, and it was worth more than any architecture change I tried.

The scaler is fitted **transductively**, on train and test together. That is legitimate
here: it uses no labels, only the feature distribution, and the test features are given.

I also reimplemented a public winning solution's "fractal tree" network — recursive
branching to depth 5, leaves concatenated — to see whether the exotic topology was doing
the work. It scored in line with the plain heavy net. The preprocessing was the winner's
real edge, not the architecture.

In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(SEED)

# RankGauss fitted on train+test together -- distribution only, no labels.
rg = QuantileTransformer(output_distribution="normal", n_quantiles=1000, random_state=SEED)
imp = SimpleImputer(strategy="mean")
both = np.vstack([X.values, X_test.values])
both = rg.fit_transform(imp.fit_transform(both))
Xs, Xs_test = both[:len(X)], both[len(X):]

def heavy_net(input_dim):
    m = models.Sequential([layers.Input(shape=(input_dim,))])
    for _ in range(5):
        m.add(layers.Dense(512, activation="swish"))
        m.add(layers.Dropout(0.6))
    m.add(layers.Dense(1, activation="sigmoid"))
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
    return m

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros(len(Xs))
nn_test = np.zeros(len(Xs_test))
nn_fold_aucs = []

for k, (tr_i, va_i) in enumerate(cv.split(Xs, y, groups), 1):
    model = heavy_net(Xs.shape[1])
    es = callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=15,
                                 restore_best_weights=True, verbose=0)
    rlr = callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5,
                                      patience=6, verbose=0)
    model.fit(Xs[tr_i], y[tr_i], validation_data=(Xs[va_i], y[va_i]),
              epochs=100, batch_size=128, callbacks=[es, rlr], verbose=0)
    oof[va_i] = model.predict(Xs[va_i], verbose=0).ravel()
    nn_test += model.predict(Xs_test, verbose=0).ravel() / 5
    nn_fold_aucs.append(roc_auc_score(y[va_i], oof[va_i]))
    print(f"fold {k}: {nn_fold_aucs[-1]:.5f}")

# Two different numbers, and the difference is not overfitting.
# Each fold holds out a different product, so the folds' predicted probabilities sit on
# slightly different scales; pooling them before ranking costs ~0.009 of AUC. The
# mean-of-folds is the figure comparable to a random-split CV.
print(f"\nmean of fold AUCs : {np.mean(nn_fold_aucs):.5f}  (std {np.std(nn_fold_aucs):.5f})")
print(f"pooled OOF AUC    : {roc_auc_score(y, oof):.5f}")

2026-09-03 15:45:51.258644: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788439551.271116   84850 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788439551.275069   84850 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788439551.286320   84850 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788439551.286336   84850 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788439551.286338   84850 computation_placer.cc:177] computation placer alr

2026-09-03 15:45:53.037601: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-09-03 15:45:53.037623: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:167] env: CUDA_VISIBLE_DEVICES=""
2026-09-03 15:45:53.037629: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] CUDA_VISIBLE_DEVICES is set to an empty string - this hides all GPUs from CUDA
2026-09-03 15:45:53.037632: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-09-03 15:45:53.037636: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: sherif-A520M-S2H
2026-09-03 15:45:53.037639: I external/local_xla/xla/stream

fold 1: 0.59128


fold 2: 0.58608


fold 3: 0.59870


fold 4: 0.58970


fold 5: 0.59440

mean of fold AUCs : 0.59203  (std 0.00428)
pooled OOF AUC    : 0.58658


## 9. Blend on ranks, not probabilities

AUC only cares about ordering, so the members are blended by converting each prediction
vector to ranks and averaging. This sidesteps the problem that a well-calibrated logistic
regression and an uncalibrated neural net live on different probability scales —
averaging those directly lets the wider distribution dominate.

Final submission: a 50/50 rank average of the RankGauss MLP and an L1 logistic regression.

In [11]:
lr = LogisticRegression(penalty="l1", C=0.01, solver="liblinear", random_state=SEED)
lr.fit(Xs, y)
lr_test = lr.predict_proba(Xs_test)[:, 1]

lr_oof = np.zeros(len(Xs))
for tr_i, va_i in cv.split(Xs, y, groups):
    m = LogisticRegression(penalty="l1", C=0.01, solver="liblinear", random_state=SEED)
    m.fit(Xs[tr_i], y[tr_i])
    lr_oof[va_i] = m.predict_proba(Xs[va_i])[:, 1]

print(f"L1 logistic regression, grouped OOF : {roc_auc_score(y, lr_oof):.5f}")
print(f"RankGauss MLP,          grouped OOF : {roc_auc_score(y, oof):.5f}")

blend_oof = (rankdata(oof) + rankdata(lr_oof)) / (2 * len(oof))
print(f"50/50 rank average,     grouped OOF : {roc_auc_score(y, blend_oof):.5f}")

L1 logistic regression, grouped OOF : 0.58834
RankGauss MLP,          grouped OOF : 0.58658
50/50 rank average,     grouped OOF : 0.58956


In [12]:
final = (rankdata(nn_test) + rankdata(lr_test)) / (2 * len(nn_test))

# Ranks are uniform on [0,1], so the mean is ~0.5 by construction; check the
# probability-scale members instead, which is where a preprocessing bug would show.
check_submission(nn_test, expected=0.21, tol=0.06)
check_submission(lr_test, expected=0.21, tol=0.06)

submission = pd.DataFrame({"id": test["id"], "failure": final})
submission.to_csv("submission_rank_blend.csv", index=False)
print(submission.head())
print(f"\nwrote submission_rank_blend.csv  ({len(submission)} rows)")

prediction mean 0.2022 -- ok
prediction mean 0.2159 -- ok


      id   failure
0  26570  0.474055
1  26571  0.198628
2  26572  0.379037
3  26573  0.357690
4  26574  0.972395

wrote submission_rank_blend.csv  (20775 rows)


## What I learned

**Design the validation split before touching a model — then keep using it.** I built
grouped CV because the `product_code` disjointness demanded it, then dropped it three
cells later without noticing. Re-running under it afterwards showed the conclusions were
safe — but I only know that because I went back and checked. Reporting numbers from a
split you didn't use is a claim about an analysis you didn't run, true or not.

**Weak signal inverts the usual model ranking.** "Try gradient boosting first" is good
default advice and it was wrong here. Low signal-to-noise plus a near-linear relationship
favours a heavily regularised linear model, and `C ≈ 0.007` says so explicitly.

**Preprocessing must be identical on both sides — and asserted, not assumed.** The
`loading` bug was invisible in every metric except the mean predicted probability.

**Read the noise floor.** With a fold std around 0.007, differences under ~0.014 are not
real. That killed several "improvements" and saved a lot of time — and it means my 6th
place and the winner's 1st are separated by less than a third of one standard deviation.

**Rank-average when the metric is rank-based.** Free robustness, no calibration required.

**Reimplementing a winning solution is a good diagnostic.** Copying the fractal
architecture and finding it *not* better told me the real edge was preprocessing. That is
more useful than a score.